# **BASE HR**

Este libro tiene como objetivo obtener información de Banxico para construir la bases de Gastos por Staff de una empresa que inicia el año con 500 empleados.

In [1]:
pip install tabula-py                   # Se utiliza para poder leer archivos en PDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 74.5 MB/s eta 0:00:00


In [2]:
from bs4 import BeautifulSoup           # Se utiliza para hacer el Scraping del sitio Web
import requests                         # Se utiliza para solicitar el uso del URL
import pandas as pd                     # Se utiliza para la estructura de datos
import matplotlib.pyplot as plt         # Se utiliza para visualizar los datos
import numpy as np
import tabula
import re
import os

In [3]:
url = "https://transparencia.banxico.org.mx/documentos/%7B34AAE447-A449-1C4C-9F69-315579D4FBCB%7D.pdf"       # guarda en una variable la URL a la que ser hará el scraping
req = requests.get(url, verify=False)

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'transparencia.banxico.org.mx'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [4]:
file_path = url
dfs = tabula.read_pdf(file_path,pages='all')                #Se lee la tabla extraida
if dfs:
    df = dfs[0]
    print(df)

    df.to_csv("tabla_extraida.csv", index=False)            #Se guarda en base
    print("Tabla guardada como tabla_extraida.csv")
else:
    print("No se encontraron tablas.")

                    Bandas Salariales 2022
0    (Importes de tabulador mensual bruto)
1                                      NaN
2     Clave o nivel Denominación o Importe
3               del puesto descripción del
4   (Banda Salarial) puesto Mínimo  Máximo
5                       BS-1 $5,063 $8,607
6                      BS-2 $5,571 $10,693
7          BS-3 Operativo/a $6,710 $15,649
8                      BS-4 $8,862 $20,866
9                     BS-5 $11,902 $30,123
10           BS-6 Analista $13,927 $52,161
11                    BS-7 $17,725 $65,203
12            BS-8 Jefe/a  $22,789 $78,244
13                    BS-9 $31,652 $91,284
14     BS-10 Subgerente/a $43,046 $104,324
15                  BS-11 $54,443 $117,364
16                  BS-12 $65,836 $130,406
17        BS-13 Gerente/a $77,230 $143,446
18                 BS-14 $120,332 $150,145
19                              Director/a
20                 BS-15 $128,275 $160,057
21      BS-16 Director/a $136,744 $170,624
22         

In [5]:
Base_salarial_BANXICO_data = pd.read_csv("tabla_extraida.csv")    #Se abre base para manipularla y dar el formato que se requiere

Este código tiene como objetivo manipular la base extraida y eliminar la información que no genera valor para quedarnos solo con la base salarial promedio por banda (puesto).

In [6]:
parsed_rows = []
pending_prefix_description = ""

for idx in range(5, len(Base_salarial_BANXICO_data)):
    line = str(Base_salarial_BANXICO_data.iloc[idx, 0]).strip()

    if line == 'Director/a':
        pending_prefix_description = 'Director/a'
        continue

    match = re.match(r'(BS-\d+)\s*(.*?)\s*(\$[\d,]+)\s*(\$[\d,]+)', line)

    if match:
        banda_salarial = match.group(1)
        extracted_description_from_line = match.group(2).strip()

        final_description_for_current_row = ""

        if pending_prefix_description:
            if not extracted_description_from_line:
                final_description_for_current_row = pending_prefix_description
            elif extracted_description_from_line == pending_prefix_description:
                final_description_for_current_row = pending_prefix_description
            else:
                final_description_for_current_row = f"{pending_prefix_description} {extracted_description_from_line}"
        else:
            final_description_for_current_row = extracted_description_from_line

        importe_minimo = match.group(3)
        importe_maximo = match.group(4)

        parsed_rows.append([
            banda_salarial,
            final_description_for_current_row.strip(),
            importe_minimo,
            importe_maximo
        ])
        pending_prefix_description = ""

result_df = pd.DataFrame(parsed_rows, columns=['banda_salarial', 'descripción del puesto', 'importe_mínimo', 'importe_máximo'])

result_df.rename(columns={
    'importe_mínimo': 'importe mínimo',
    'importe_máximo': 'importe máximo'
}, inplace=True)

final_columns_order = ['descripción del puesto', 'banda_salarial', 'importe máximo', 'importe mínimo']
final_df_salaries = result_df[final_columns_order].copy()

final_df_salaries['importe máximo_numeric'] = final_df_salaries['importe máximo'].replace({"\$": '', ',': ''}, regex=True).astype(float)
final_df_salaries['importe mínimo_numeric'] = final_df_salaries['importe mínimo'].replace({"\$": '', ',': ''}, regex=True).astype(float)

final_df_salaries['Salario Mensual Promedio'] = (final_df_salaries['importe máximo_numeric'] + final_df_salaries['importe mínimo_numeric']) / 2

display(final_df_salaries[['banda_salarial', 'Salario Mensual Promedio']])

<>:50: SyntaxWarning: invalid escape sequence '\$'
<>:51: SyntaxWarning: invalid escape sequence '\$'
<>:50: SyntaxWarning: invalid escape sequence '\$'
<>:51: SyntaxWarning: invalid escape sequence '\$'
/tmp/ipykernel_1429/4027138698.py:50: SyntaxWarning: invalid escape sequence '\$'
  final_df_salaries['importe máximo_numeric'] = final_df_salaries['importe máximo'].replace({"\$": '', ',': ''}, regex=True).astype(float)
/tmp/ipykernel_1429/4027138698.py:51: SyntaxWarning: invalid escape sequence '\$'
  final_df_salaries['importe mínimo_numeric'] = final_df_salaries['importe mínimo'].replace({"\$": '', ',': ''}, regex=True).astype(float)


,banda_salarial,Salario Mensual Promedio
0,BS-1,6835.0
1,BS-2,8132.0
2,BS-3,11179.5
3,BS-4,14864.0
4,BS-5,21012.5
5,BS-6,33044.0
6,BS-7,41464.0
7,BS-8,50516.5
8,BS-9,61468.0
9,BS-10,73685.0


In [7]:
# Guarda una nueva base limpia.
final_df_salaries = final_df_salaries.drop(columns=['descripción del puesto', 'importe máximo', 'importe mínimo', 'importe máximo_numeric', 'importe mínimo_numeric'], errors='ignore')
final_df_salaries.to_csv("Base_limpia_final.csv", index=False, encoding='utf-8')
df = pd.read_csv("Base_limpia_final.csv")


In [8]:
Base_salarial_data = pd.read_csv("Base_limpia_final.csv")

Generación del gasto mensual y acumulado a un año por los diferentes componentes del Staff Cost basado en el salario mensual promedio obtenido inicialmente. Se guarda un archivo por componente.

In [9]:
# Definición de los componentes (parámetros)

Wages_and_Salaries = 1          #Wages & Salaries
Profit_Share = 0.20             #Profit Share (PTU)
Social_Security = 0.10          #Social Security
Employe_Benefit = 400           #Employe Benefit
Social_Taxes  = 0.16            #Social Taxes

parametros = ["Wages_and_Salaries", "Profit_Share", "Social_Security", "Employe_Benefit", "Social_Taxes"] # Por ejemplo, multiplicadores de precio


In [10]:
Base_salarial_data = pd.read_csv("Base_limpia_final.csv")
display(Base_salarial_data)

,banda_salarial,Salario Mensual Promedio
0,BS-1,6835.0
1,BS-2,8132.0
2,BS-3,11179.5
3,BS-4,14864.0
4,BS-5,21012.5
5,BS-6,33044.0
6,BS-7,41464.0
7,BS-8,50516.5
8,BS-9,61468.0
9,BS-10,73685.0


In [11]:
import os

employe_benefit_idx = 3

output_dir = "output_data"

for i, param_name in enumerate(parametros):
    param_value = globals()[param_name]

    months = ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio', 'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']    # Create columns for each month (January to December)

    current_df = final_df_salaries[['banda_salarial', 'Salario Mensual Promedio']].copy()

    if i != employe_benefit_idx:
        for month in months:
            current_df[month] = current_df['Salario Mensual Promedio'] * param_value
    else :
        for month in months:
            current_df[month] = param_value

    current_df['Dic YTD'] = current_df[months].sum(axis=1)
    current_df['Cost Unit Managment'] = param_name

    cols = current_df.columns.tolist()
    desired_order = ['banda_salarial', 'Cost Unit Managment', 'Salario Mensual Promedio'] + months + ['Dic YTD']
    ordered_cols = [col for col in desired_order if col in cols]
    current_df = current_df[ordered_cols]

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    nombre_archivo = f"{output_dir}/Base_{param_name}.csv"
    current_df.to_csv(nombre_archivo, index=False, encoding='utf-8')

Generación del gasto mensual y acumulado a un año del componente del Staff Cost en el que no interviene en el salario mensual promedio. Se guarda un archivo con este componente.

In [12]:
import os

Aguinaldo = 0.5                 #Aguinaldo

months = ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio', 'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']    # Create columns for each month (January to December)

current_df = Base_salarial_data[['banda_salarial', 'Salario Mensual Promedio']].copy()

for month_name in months:
    if month_name != 'Diciembre':
        current_df[month_name] = current_df['Salario Mensual Promedio'] * 0
    else:
        current_df[month_name] = current_df['Salario Mensual Promedio'] * Aguinaldo

current_df['Dic YTD'] = current_df[months].sum(axis=1)
current_df['Cost Unit Managment'] = 'Aguinaldo'

cols = current_df.columns.tolist()
desired_order = ['banda_salarial', 'Cost Unit Managment', 'Salario Mensual Promedio'] + months + ['Dic YTD']
ordered_cols = [col for col in desired_order if col in cols]
current_df = current_df[ordered_cols]

output_dir = "output_data"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

file_path = os.path.join(output_dir, "Base_Aguinaldo.csv")
current_df.to_csv(file_path, index=False, encoding='utf-8')


Generación aleatoria del número de empleados asumiendo que imiciamos el año con 500 empleados y una tasa aleatoria por banda y por mes del 2% y asumiendo que la banda entre mayor es la banda menor es el número de empleados y que nunca hay una banda sin empleados. Se guarda la base de FTEs (empleados).

In [13]:
import numpy as np

personnel_base_data = Base_salarial_data.copy()
personnel_base_data['banda_num'] = personnel_base_data['banda_salarial'].str.extract('(\\d+)').astype(int)
sorted_personnel_df = personnel_base_data.sort_values(by='banda_num', ascending=False).reset_index(drop=True)

total_employees_target = 500
employees_data = []

bs17_row = sorted_personnel_df.iloc[0]
employees_data.append({
    'banda_salarial': bs17_row['banda_salarial'],
    'Salario Mensual Promedio': bs17_row['Salario Mensual Promedio'],
    'num_employees': 1
})

remaining_employees_to_distribute = total_employees_target - 1

other_bands_df = sorted_personnel_df.iloc[1:].copy()

max_other_band_num = other_bands_df['banda_num'].max() # Should be 16
other_bands_df['weight'] = max_other_band_num - other_bands_df['banda_num'] + 1

probabilities = other_bands_df['weight'] / other_bands_df['weight'].sum()

assigned_employees = np.round(probabilities * remaining_employees_to_distribute).astype(int)

current_sum = assigned_employees.sum()
diff = remaining_employees_to_distribute - current_sum

if diff != 0:
    for i in range(abs(diff)):
        if diff > 0:
            assigned_employees.iloc[-(i % len(assigned_employees)) - 1] += 1
        else:
            if assigned_employees.iloc[-(i % len(assigned_employees)) - 1] > 0:
                assigned_employees.iloc[-(i % len(assigned_employees)) - 1] -= 1
            else:
                pass

final_remaining_sum = assigned_employees.sum()

other_bands_df['num_employees'] = assigned_employees

for index, row in other_bands_df.iterrows():
    employees_data.append({
        'banda_salarial': row['banda_salarial'],
        'Salario Mensual Promedio': row['Salario Mensual Promedio'],
        'num_employees': max(0, row['num_employees'])
    })

base_personal_df = pd.DataFrame(employees_data)

base_personal_df = base_personal_df.sort_values(by='banda_salarial', key=lambda x: x.str.extract('(\\d+)')[0].astype(int)).reset_index(drop=True)

display(base_personal_df)
print(f"Total employees generated: {base_personal_df['num_employees'].sum()}")

base_personal_df.to_csv("Base_FTEs_random_500.csv", index=False, encoding='utf-8')
print("Personnel base with random distribution saved as Base_FTEs_random_500.csv")

,banda_salarial,Salario Mensual Promedio,num_employees
0,BS-1,6835.0,59
1,BS-2,8132.0,55
2,BS-3,11179.5,51
3,BS-4,14864.0,48
4,BS-5,21012.5,44
5,BS-6,33044.0,40
6,BS-7,41464.0,37
7,BS-8,50516.5,33
8,BS-9,61468.0,29
9,BS-10,73685.0,26


Total employees generated: 500
Personnel base with random distribution saved as Base_FTEs_random_500.csv


In [14]:
import pandas as pd
import numpy as np
import os

monthly_personnel_df_base = base_personal_df.copy()
monthly_personnel_df_base['proportion'] = monthly_personnel_df_base['num_employees'] / monthly_personnel_df_base['num_employees'].sum()

months = ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio', 'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']

final_monthly_personnel_df = base_personal_df[['banda_salarial', 'Salario Mensual Promedio']].copy()
final_monthly_personnel_df[months[0]] = base_personal_df['num_employees']

current_total_employees = base_personal_df['num_employees'].sum()
num_bands = len(base_personal_df)

for i in range(1, len(months)):
    month_name = months[i]

    fluctuation = np.random.randint(-10, 11)

    new_total_employees = current_total_employees + fluctuation

    new_total_employees = max(new_total_employees, num_bands)

    proportional_employees_raw = monthly_personnel_df_base['proportion'] * new_total_employees

    employees_this_month = np.maximum(1, np.floor(proportional_employees_raw)).astype(int)

    current_sum_after_min_1 = employees_this_month.sum()
    diff = new_total_employees - current_sum_after_min_1

    if diff != 0:
        employees_series = pd.Series(employees_this_month.copy(), index=monthly_personnel_df_base.index)

        if diff > 0:
            fractional_parts = proportional_employees_raw - np.floor(proportional_employees_raw)
            sorted_indices_for_adding = fractional_parts.argsort()[::-1]

            k = 0
            while diff > 0:
                idx_in_df = sorted_indices_for_adding[k % num_bands]
                employees_series.loc[idx_in_df] += 1
                diff -= 1
                k += 1

        elif diff < 0:
            sorted_indices_by_count_desc = employees_series.sort_values(ascending=False).index

            k = 0
            while diff < 0:
                idx_in_df = sorted_indices_by_count_desc[k % num_bands]
                if employees_series.loc[idx_in_df] > 1:
                    employees_series.loc[idx_in_df] -= 1
                    diff += 1
                k += 1
                if k > num_bands * 2:
                    break

        employees_this_month = employees_series.values

    final_monthly_personnel_df[month_name] = employees_this_month
    current_total_employees = new_total_employees

final_monthly_personnel_df['Dic YTD'] = final_monthly_personnel_df['Diciembre']

final_monthly_personnel_df['Cost Unit Managment'] = 'FTEs'

desired_order = ['banda_salarial', 'Cost Unit Managment'] + months + ['Dic YTD']
if 'Salario Mensual Promedio' in final_monthly_personnel_df.columns:
    final_monthly_personnel_df = final_monthly_personnel_df.drop(columns=['Salario Mensual Promedio'])

final_monthly_personnel_df = final_monthly_personnel_df[desired_order]

output_dir = "output_data"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

file_path = os.path.join(output_dir, "Base_FTEs.csv")
final_monthly_personnel_df.to_csv(file_path, index=False, encoding='utf-8')

print(f"Monthly FTEs data saved to {file_path}")
display(final_monthly_personnel_df.head())

Monthly FTEs data saved to output_data/Base_FTEs.csv


,banda_salarial,Cost Unit Managment,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre,Dic YTD
0,BS-1,FTEs,59,59,59,59,59,59,59,59,59,57,58,57,57
1,BS-2,FTEs,55,55,55,55,55,55,55,55,55,55,54,53,53
2,BS-3,FTEs,51,51,51,51,51,51,51,51,51,51,50,49,49
3,BS-4,FTEs,48,48,48,48,48,48,48,48,48,48,46,46,46
4,BS-5,FTEs,44,44,44,44,44,44,44,44,44,44,42,44,44


Se genera una Base completa con todos los gastos generados por Staff, se eliminan elementos duplicados y se guarda como final.

In [15]:
import pandas as pd
import os

output_dir = "output_data"

all_files = os.listdir(output_dir)

csv_files_to_include = [
    "Base_Aguinaldo.csv",
    "Base_Employe_Benefit.csv",
    "Base_Profit_Share.csv",
    "Base_Social_Security.csv",
    "Base_Social_Taxes.csv",
    "Base_Wages_and_Salaries.csv"
]

csv_files_to_combine = [
    os.path.join(output_dir, f)
    for f in all_files
    if f in csv_files_to_include
]

list_of_dfs = []

for file in csv_files_to_combine:
    df = pd.read_csv(file)
    list_of_dfs.append(df)

Base_Completa_HR = pd.concat(list_of_dfs, ignore_index=True)

if 'Salario Mensual Promedio' in Base_Completa_HR.columns:
    Base_Completa_HR = Base_Completa_HR.drop(columns=['Salario Mensual Promedio'])

print(f"Shape of DataFrame before removing duplicates: {Base_Completa_HR.shape}")

Base_Completa_HR.drop_duplicates(inplace=True)

print(f"Shape of DataFrame after removing duplicates: {Base_Completa_HR.shape}")

output_file_path = os.path.join(output_dir, "Base_Completa_HR.csv")
Base_Completa_HR.to_csv(output_file_path, index=False, encoding='utf-8')

print(f"Combined DataFrame saved to {output_file_path}")
display(Base_Completa_HR.head())

Shape of DataFrame before removing duplicates: (102, 15)
Shape of DataFrame after removing duplicates: (102, 15)
Combined DataFrame saved to output_data/Base_Completa_HR.csv


,banda_salarial,Cost Unit Managment,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre,Dic YTD
0,BS-1,Aguinaldo,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3417.50,3417.50
1,BS-2,Aguinaldo,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4066.00,4066.00
2,BS-3,Aguinaldo,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5589.75,5589.75
3,BS-4,Aguinaldo,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7432.00,7432.00
4,BS-5,Aguinaldo,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10506.25,10506.25


Genera la Base del gasto por Staff por de acuerdo al número de FTEs por banda y por mes

In [16]:
import pandas as pd
import os

output_dir = "output_data"

df_expenses = pd.read_csv(os.path.join(output_dir, "Base_Completa_HR.csv"))

df_ftes_raw = pd.read_csv(os.path.join(output_dir, "Base_FTEs.csv"))

months = ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio', 'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']

ftes_cols = ['banda_salarial'] + months
df_ftes = df_ftes_raw[ftes_cols].copy()

ftes_month_mapping = {month: f'{month}_FTEs' for month in months}
df_ftes = df_ftes.rename(columns=ftes_month_mapping)

list_of_multiplied_dfs = []

cost_types = df_expenses['Cost Unit Managment'].unique()

for cost_type in cost_types:
    current_cost_df = df_expenses[df_expenses['Cost Unit Managment'] == cost_type].copy()

    merged_df = pd.merge(current_cost_df, df_ftes, on='banda_salarial', how='left')

    for month in months:
        expense_col = month
        fte_col = f'{month}_FTEs'
        multiplied_col = f'{month}'

        merged_df[multiplied_col] = merged_df[expense_col] * merged_df[fte_col]

    merged_df['Dic YTD'] = merged_df[months].sum(axis=1)

    cols_to_keep = ['banda_salarial', 'Cost Unit Managment'] + months + ['Dic YTD']
    final_df_for_cost_type = merged_df[cols_to_keep]

    list_of_multiplied_dfs.append(final_df_for_cost_type)

final_combined_hr_ftes_df = pd.concat(list_of_multiplied_dfs, ignore_index=True)

print(f"Shape of DataFrame before removing duplicates: {final_combined_hr_ftes_df.shape}")


final_combined_hr_ftes_df.drop_duplicates(inplace=True)

print(f"Shape of DataFrame after removing duplicates: {final_combined_hr_ftes_df.shape}")

output_file_path = os.path.join(output_dir, "Base_Staff_Cost_2025.csv")
final_combined_hr_ftes_df.to_csv(output_file_path, index=False, encoding='utf-8')

print(f"Combined HR and FTEs DataFrame saved to {output_file_path}")
print("Head of the new combined DataFrame:")
display(final_combined_hr_ftes_df.head())

Shape of DataFrame before removing duplicates: (102, 15)
Shape of DataFrame after removing duplicates: (102, 15)
Combined HR and FTEs DataFrame saved to output_data/Base_Staff_Cost_2025.csv
Head of the new combined DataFrame:


,banda_salarial,Cost Unit Managment,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre,Dic YTD
0,BS-1,Aguinaldo,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,194797.50,194797.50
1,BS-2,Aguinaldo,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,215498.00,215498.00
2,BS-3,Aguinaldo,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,273897.75,273897.75
3,BS-4,Aguinaldo,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,341872.00,341872.00
4,BS-5,Aguinaldo,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,462275.00,462275.00


Por último generamos una base que incluya los FTEs y será nuestra base para la proyección del FRP del 2026.

In [17]:
import pandas as pd
import os

output_dir = "output_data"

all_files = os.listdir(output_dir)

csv_files_to_include = [
    "Base_Staff_Cost_2025.csv",
    "Base_FTEs.csv"
]

csv_files_to_combine = [
    os.path.join(output_dir, f)
    for f in all_files
    if f in csv_files_to_include
]

list_of_dfs = []

for file in csv_files_to_combine:
    df = pd.read_csv(file)
    list_of_dfs.append(df)

Base_FRP_2026 = pd.concat(list_of_dfs, ignore_index=True)

if 'Salario Mensual Promedio' in Base_FRP_2026.columns:
    Base_FRP_2026 = Base_FRP_2026.drop(columns=['Salario Mensual Promedio'])

print(f"Shape of DataFrame before removing duplicates: {Base_FRP_2026.shape}")

Base_FRP_2026.drop_duplicates(inplace=True)

print(f"Shape of DataFrame after removing duplicates: {Base_FRP_2026.shape}")

output_file_path = os.path.join(output_dir, "Base_FRP_2026.csv")
Base_FRP_2026.to_csv(output_file_path, index=False, encoding='utf-8')

print(f"Combined DataFrame saved to {output_file_path}")
display(Base_FRP_2026.head())

Shape of DataFrame before removing duplicates: (119, 15)
Shape of DataFrame after removing duplicates: (119, 15)
Combined DataFrame saved to output_data/Base_FRP_2026.csv


,banda_salarial,Cost Unit Managment,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre,Dic YTD
0,BS-1,FTEs,59.0,59.0,59.0,59.0,59.0,59.0,59.0,59.0,59.0,57.0,58.0,57.0,57.0
1,BS-2,FTEs,55.0,55.0,55.0,55.0,55.0,55.0,55.0,55.0,55.0,55.0,54.0,53.0,53.0
2,BS-3,FTEs,51.0,51.0,51.0,51.0,51.0,51.0,51.0,51.0,51.0,51.0,50.0,49.0,49.0
3,BS-4,FTEs,48.0,48.0,48.0,48.0,48.0,48.0,48.0,48.0,48.0,48.0,46.0,46.0,46.0
4,BS-5,FTEs,44.0,44.0,44.0,44.0,44.0,44.0,44.0,44.0,44.0,44.0,42.0,44.0,44.0


In [20]:
import pandas as pd
import os

output_dir = "output_data"

# Load the Base_FRP_2026.csv file
file_path = os.path.join(output_dir, "Base_FRP_2026.csv")
Base_FRP_2026_filtered = pd.read_csv(file_path)

# Filter out rows where 'Cost Unit Managment' is not 'Wages_and_Salaries' or 'FTEs'
Base_FRP_2026_filtered = Base_FRP_2026_filtered[Base_FRP_2026_filtered['Cost Unit Managment'].isin(['Wages_and_Salaries', 'FTEs'])]

# Define the months to drop
months_to_drop = ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio', 'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Dic YTD']

# Drop the monthly columns
Base_FRP_2026_filtered = Base_FRP_2026_filtered.drop(columns=months_to_drop, errors='ignore')

print("Filtered Base_FRP_2026 DataFrame:")
display(Base_FRP_2026_filtered.head())

# Optionally, save the filtered DataFrame
output_filtered_file_path = os.path.join(output_dir, "Base_FRP_2026_filtered.csv")
Base_FRP_2026_filtered.to_csv(output_filtered_file_path, index=False, encoding='utf-8')
print(f"Filtered DataFrame saved to {output_filtered_file_path}")

Filtered Base_FRP_2026 DataFrame:


,banda_salarial,Cost Unit Managment,Diciembre
0,BS-1,FTEs,57.0
1,BS-2,FTEs,53.0
2,BS-3,FTEs,49.0
3,BS-4,FTEs,46.0
4,BS-5,FTEs,44.0


Filtered DataFrame saved to output_data/Base_FRP_2026_filtered.csv


In [32]:
import pandas as pd
import os

output_dir = "output_data"
file_path = os.path.join(output_dir, "Base_FRP_2026_filtered.csv")
Base_FRP_2026_filtered = pd.read_csv(file_path)

inflation_rate = 0.0317

# Filter for 'Wages_and_Salaries' rows
wages_salaries_df = Base_FRP_2026_filtered[Base_FRP_2026_filtered['Cost Unit Managment'] == 'Wages_and_Salaries'].copy()

# Apply inflation to the 'Diciembre' column
wages_salaries_df['Diciembre'] = wages_salaries_df['Diciembre'] * (1 + inflation_rate)

# Update the 'Cost Unit Managment' to indicate inflation
wages_salaries_df['Cost Unit Managment'] = 'Wages_and_Salaries_Inflated'

# Concatenate the original filtered DataFrame with the new inflated wages_salaries DataFrame
Base_FRP_2026_updated = pd.concat([Base_FRP_2026_filtered, wages_salaries_df], ignore_index=True)

# Rename the 'Diciembre' column to 'Enero' as requested by the user
Base_FRP_2026_updated.rename(columns={'Diciembre': 'Enero 2026'}, inplace=True)

print("Base_FRP_2026 DataFrame with inflation applied:")
display(Base_FRP_2026_updated.head())

# Optionally, save the updated DataFrame
output_updated_file_path = os.path.join(output_dir, "Base_FRP_2026_inflated_wages.csv")
Base_FRP_2026_updated.to_csv(output_updated_file_path, index=False, encoding='utf-8')
print(f"Updated DataFrame saved to {output_updated_file_path}")

Base_FRP_2026 DataFrame with inflation applied:


,banda_salarial,Cost Unit Managment,Enero 2026
0,BS-1,FTEs,57.0
1,BS-2,FTEs,53.0
2,BS-3,FTEs,49.0
3,BS-4,FTEs,46.0
4,BS-5,FTEs,44.0


Updated DataFrame saved to output_data/Base_FRP_2026_inflated_wages.csv


In [38]:
import pandas as pd
import os

output_dir = "output_data"
file_path = os.path.join(output_dir, "Base_FRP_2026_inflated_wages.csv")
Base_FRP_2026_combined = pd.read_csv(file_path)

# Parameters for 2026 calculations (assuming same as previous year)
Profit_Share = 0.20
Social_Security = 0.10
Social_Taxes = 0.16

# Extract the base salaries for 2026 (Wages_and_Salaries_Inflated)
# This df already contains the inflated wages and is correctly named 'Wages_and_Salaries_Inflated'
base_salaries_2026_df = Base_FRP_2026_combined[Base_FRP_2026_combined['Cost Unit Managment'] == 'Wages_and_Salaries_Inflated'].copy()
base_salaries_2026_df = base_salaries_2026_df[['banda_salarial', 'Enero 2026']]
base_salaries_2026_df.rename(columns={'Enero 2026': 'Base Salary 2026'}, inplace=True)

new_components_list = []

# Calculate Profit Share for 2026
profit_share_2026_df = base_salaries_2026_df.copy()
profit_share_2026_df['Enero 2026'] = profit_share_2026_df['Base Salary 2026'] * Profit_Share
profit_share_2026_df['Cost Unit Managment'] = 'Profit_Share_2026'
new_components_list.append(profit_share_2026_df[['banda_salarial', 'Cost Unit Managment', 'Enero 2026']])

# Calculate Social Security for 2026
social_security_2026_df = base_salaries_2026_df.copy()
social_security_2026_df['Enero 2026'] = social_security_2026_df['Base Salary 2026'] * Social_Security
social_security_2026_df['Cost Unit Managment'] = 'Social_Security_2026'
new_components_list.append(social_security_2026_df[['banda_salarial', 'Cost Unit Managment', 'Enero 2026']])

# Calculate Social Taxes for 2026
social_taxes_2026_df = base_salaries_2026_df.copy()
social_taxes_2026_df['Enero 2026'] = social_taxes_2026_df['Base Salary 2026'] * Social_Taxes
social_taxes_2026_df['Cost Unit Managment'] = 'Social_Taxes_2026'
new_components_list.append(social_taxes_2026_df[['banda_salarial', 'Cost Unit Managment', 'Enero 2026']])

# Filter out original 'Wages_and_Salaries' and 'FTEs' from the Base_FRP_2026_combined before concatenation
#Base_FRP_2026_cleaned = Base_FRP_2026_combined[
 #   (Base_FRP_2026_combined['Cost Unit Managment'] != 'Wages_and_Salaries') &
  #  (Base_FRP_2026_combined['Cost Unit Managment'] != 'FTEs')
#].copy()

# Concatenate new components with the existing Base_FRP_2026_cleaned
FRP_2026_Projected = pd.concat([Base_FRP_2026_cleaned] + new_components_list, ignore_index=True)

# Rename 'Wages_and_Salaries_Inflated' to 'Wages_and_Salaries'
#Base_FRP_2026_final['Cost Unit Managment'] = Base_FRP_2026_final['Cost Unit Managment'].replace('Wages_and_Salaries_Inflated', 'Wages_and_Salaries')

print("Base_FRP_2026 DataFrame with additional components for Enero 2026:")
display(Base_FRP_2026_final.head())

# Save the final DataFrame
output_final_file_path = os.path.join(output_dir, "FRP_2026_Projections.csv")
Base_FRP_2026_final.to_csv(output_final_file_path, index=False, encoding='utf-8')
print(f"Final DataFrame saved to {output_final_file_path}")

Base_FRP_2026 DataFrame with additional components for Enero 2026:


,banda_salarial,Cost Unit Managment,Enero 2026
0,BS-1,Wages_and_Salaries,401945.16150
1,BS-2,Wages_and_Salaries,444658.57320
2,BS-3,Wages_and_Salaries,565160.61735
3,BS-4,Wages_and_Salaries,705418.68480
4,BS-5,Wages_and_Salaries,953858.23500


Final DataFrame saved to output_data/FRP_2026_Projections.csv
